In [1]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

Saving stress_audio_pipeline.joblib to stress_audio_pipeline.joblib
Saving stress_text_pipeline.joblib to stress_text_pipeline.joblib
Saving stress_video_config.json to stress_video_config.json
Saving stress_video_resnet18.pt to stress_video_resnet18.pt
Uploaded: ['stress_audio_pipeline.joblib', 'stress_text_pipeline.joblib', 'stress_video_config.json', 'stress_video_resnet18.pt']


In [2]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

Saving 01-01-01-01-01-01-01.avi to 01-01-01-01-01-01-01.avi
Saving 01-01-01-01-02-01-01.avi to 01-01-01-01-02-01-01.avi
Saving 01-01-01-01-02-02-01.avi to 01-01-01-01-02-02-01.avi
Saving 01-01-02-01-01-01-01.avi to 01-01-02-01-01-01-01.avi
Saving 01-01-02-01-01-02-01.avi to 01-01-02-01-01-02-01.avi
Saving 01-01-02-01-02-01-01.avi to 01-01-02-01-02-01-01.avi
Saving 01-01-02-01-02-02-01.avi to 01-01-02-01-02-02-01.avi
Saving 01-01-02-02-01-01-01.avi to 01-01-02-02-01-01-01.avi
Saving 01-01-02-02-01-02-01.avi to 01-01-02-02-01-02-01.avi
Saving 01-01-02-02-02-01-01.avi to 01-01-02-02-02-01-01.avi
Saving 01-01-02-02-02-02-01.avi to 01-01-02-02-02-02-01.avi
Saving 01-01-03-01-01-01-01.avi to 01-01-03-01-01-01-01.avi
Saving 01-01-03-01-01-02-01.avi to 01-01-03-01-01-02-01.avi
Saving 01-01-03-01-02-01-01.avi to 01-01-03-01-02-01-01.avi
Saving 01-01-03-01-02-02-01.avi to 01-01-03-01-02-02-01.avi
Saving 01-01-03-02-01-01-01.avi to 01-01-03-02-01-01-01.avi
Saving 01-01-03-02-01-02-01.avi to 01-01

In [47]:
import os
import glob
import json
import numpy as np
import joblib
import librosa
import cv2
import torch

from PIL import Image
from torchvision import transforms

In [48]:
import shutil

os.makedirs("audio", exist_ok=True)
os.makedirs("video", exist_ok=True)

video_exts = (".avi",".mp4",".mov",".mkv")

moved_audio = 0
moved_video = 0

for f in os.listdir("."):
    if os.path.isfile(f):
        if f.lower().endswith(".wav"):
            shutil.move(f, os.path.join("audio", f))
            moved_audio += 1
        elif f.lower().endswith(video_exts):
            shutil.move(f, os.path.join("video", f))
            moved_video += 1

print("Moved audio:", moved_audio)
print("Moved video:", moved_video)

print("Audio files:", len(glob.glob("audio/*.wav")))
print("Video files:", len(glob.glob("video/*")))

Moved audio: 0
Moved video: 0
Audio files: 61
Video files: 103


In [49]:
TEXT_MODEL  = "stress_text_pipeline.joblib"
AUDIO_MODEL = "stress_audio_pipeline.joblib"
VIDEO_MODEL = "stress_video_resnet18.pt"
VIDEO_CFG   = "stress_video_config.json"

# load text model
text_pipe = joblib.load(TEXT_MODEL)

# load audio model
audio_pipe = joblib.load(AUDIO_MODEL)

# load video config
with open(VIDEO_CFG,"r") as f:
    cfg = json.load(f)

IMG_SIZE   = cfg["IMG_SIZE"]
NUM_FRAMES = cfg["NUM_FRAMES"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Models loaded successfully")

Models loaded successfully


In [52]:
import torch
import torchvision.models as models
from torchvision import transforms

video_model = models.resnet18(weights=None)
video_model.fc = torch.nn.Linear(video_model.fc.in_features, 2)

state = torch.load(VIDEO_MODEL, map_location=device)

new_state = {}
for k, v in state.items():
    # 1) strip backbone.
    if k.startswith("backbone."):
        k = k.replace("backbone.", "", 1)

    # 2) rename classifier -> fc (last layer)
    if k == "classifier.weight":
        k = "fc.weight"
    elif k == "classifier.bias":
        k = "fc.bias"

    new_state[k] = v

# strict=True should work now
video_model.load_state_dict(new_state, strict=True)

video_model.to(device)
video_model.eval()

img_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

print("Video model ready ✅")

Video model ready ✅


In [53]:
def text_score(text):
    p = text_pipe.predict_proba([text])[0]

    if len(p)==2:
        return float(p[1])

    return float(np.max(p))

In [54]:
def extract_audio_features(path):

    y, sr = librosa.load(path, sr=22050)

    mfcc = librosa.feature.mfcc(y=y,sr=sr,n_mfcc=40)
    zcr  = librosa.feature.zero_crossing_rate(y)
    rms  = librosa.feature.rms(y=y)

    feat = np.hstack([
        mfcc.mean(axis=1),
        mfcc.std(axis=1),
        zcr.mean(axis=1),
        zcr.std(axis=1),
        rms.mean(axis=1),
        rms.std(axis=1)
    ])

    return feat

In [55]:
def sigmoid(x):
    return 1/(1+np.exp(-x))


def audio_score(path):

    x = extract_audio_features(path)
    x = x.reshape(1,-1)

    if hasattr(audio_pipe,"decision_function"):
        d = audio_pipe.decision_function(x)
        return float(sigmoid(d[0]))

    if hasattr(audio_pipe,"predict_proba"):
        p = audio_pipe.predict_proba(x)[0]
        return float(p[1])

    return float(audio_pipe.predict(x)[0])

In [56]:
def sample_frames(video_path):

    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total<=0:
        cap.release()
        return None

    idx = np.linspace(0,total-1,NUM_FRAMES).astype(int)

    frames=[]
    cur=0

    while True:

        ok,frame = cap.read()

        if not ok:
            break

        if cur in idx:
            frame = cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
            frames.append(frame)

        cur+=1

    cap.release()

    return frames

In [57]:
def video_score(video_path):

    frames = sample_frames(video_path)

    if frames is None:
        return None

    imgs=[img_tf(Image.fromarray(f)) for f in frames]

    x=torch.stack(imgs).to(device)

    with torch.no_grad():
        logits = video_model(x)
        p = torch.softmax(logits,dim=1)[:,1].mean().item()

    return float(p)

In [58]:
def predict_fused(text,audio_file,video_file):

    p_text  = text_score(text)
    p_audio = audio_score(audio_file)
    p_video = video_score(video_file)

    w_text  = 0.45
    w_audio = 0.25
    w_video = 0.30

    if p_video is None:
        p_final = (w_text*p_text + w_audio*p_audio)/(w_text+w_audio)
    else:
        p_final = w_text*p_text + w_audio*p_audio + w_video*p_video

    if p_final < 0.4:
        level="Low"
    elif p_final < 0.7:
        level="Moderate"
    else:
        level="High"

    return {
        "p_text":p_text,
        "p_audio":p_audio,
        "p_video":p_video,
        "p_final":p_final,
        "final_level":level
    }

In [65]:
import joblib

TEXT_MODEL_PATH = "stress_text_pipeline.joblib"   # <-- change if your file name is different

obj = joblib.load(TEXT_MODEL_PATH)
print("Loaded type:", type(obj))

def unwrap_to_predict_proba(x):
    # If it's already a pipeline/model
    if hasattr(x, "predict_proba"):
        return x

    # If saved as (something, something)
    if isinstance(x, tuple) or isinstance(x, list):
        # pick the first item that has predict_proba
        for item in x:
            if hasattr(item, "predict_proba"):
                return item
        # sometimes nested
        for item in x:
            if isinstance(item, (tuple, list)):
                for sub in item:
                    if hasattr(sub, "predict_proba"):
                        return sub

    raise ValueError("Could not find an object with predict_proba in the loaded joblib. Re-save text model as a single sklearn Pipeline.")

text_pipe = unwrap_to_predict_proba(obj)
print("Using text_pipe:", type(text_pipe), "| has predict_proba:", hasattr(text_pipe, "predict_proba"))

Loaded type: <class 'tuple'>
Using text_pipe: <class 'sklearn.linear_model._logistic.LogisticRegression'> | has predict_proba: True


In [66]:
def text_score(text: str) -> float:
    proba = text_pipe.predict_proba([text])[0]
    # binary classifier: take prob of class 1
    if len(proba) == 2:
        return float(proba[1])
    # fallback: if multi-class, take max probability (or adjust to your stress class index)
    return float(max(proba))

In [71]:
def text_score(text: str) -> float:
    proba = text_pipe.predict_proba([text])[0]   # list of 1 string
    return float(proba[1]) if len(proba) == 2 else float(max(proba))

In [73]:
def text_score(text: str) -> float:
    proba = text_pipe.predict_proba([text])[0]   # pass list of 1 string
    return float(proba[1]) if len(proba) == 2 else float(max(proba))

In [75]:
import joblib
import numpy as np

TEXT_MODEL_PATH = "stress_text_pipeline.joblib"   # put correct filename in current folder

obj = joblib.load(TEXT_MODEL_PATH)
print("Loaded:", type(obj))

def text_score(text: str) -> float:
    # Case A: you saved a full sklearn Pipeline
    if hasattr(obj, "predict_proba"):
        proba = obj.predict_proba([text])[0]   # list of 1 string
        return float(proba[1]) if len(proba) == 2 else float(np.max(proba))

    # Case B: you saved (vectorizer, classifier) tuple
    if isinstance(obj, tuple) and len(obj) >= 2:
        vec, clf = obj[0], obj[1]
        X = vec.transform([text])             # TF-IDF transform
        proba = clf.predict_proba(X)[0]
        return float(proba[1]) if len(proba) == 2 else float(np.max(proba))

    raise ValueError("Saved object is neither a Pipeline nor a (vectorizer, classifier) tuple.")

Loaded: <class 'tuple'>


In [76]:
print("calm:", text_score("Today was calm and I feel relaxed."))
print("panic:", text_score("I can't sleep, my heart is racing and I'm panicking."))

calm: 0.6738594689603875
panic: 0.567139897049096


In [77]:
print(text_pipe.classes_)

[0 1]


In [83]:
def text_score(text: str) -> float:
    proba = text_pipe.predict_proba([text])[0]
    return float(proba[0])

In [87]:
import os, glob

print("CWD:", os.getcwd())
print("Top files:", os.listdir("."))

print("\nFind joblib anywhere (recursive):")
matches = glob.glob("**/*.joblib", recursive=True)
print(matches)

CWD: /content
Top files: ['.config', 'stress_audio_pipeline.joblib', 'audio', 'stress_text_pipeline.joblib', 'stress_video_resnet18.pt', 'video', 'stress_video_config.json', 'sample_data']

Find joblib anywhere (recursive):
['stress_audio_pipeline.joblib', 'stress_text_pipeline.joblib']


In [88]:
import os, glob

print("CWD:", os.getcwd())
print("Top files:", os.listdir("."))

print("\nFind joblib anywhere (recursive):")
matches = glob.glob("**/*.joblib", recursive=True)
print(matches)

CWD: /content
Top files: ['.config', 'stress_audio_pipeline.joblib', 'audio', 'stress_text_pipeline.joblib', 'stress_video_resnet18.pt', 'video', 'stress_video_config.json', 'sample_data']

Find joblib anywhere (recursive):
['stress_audio_pipeline.joblib', 'stress_text_pipeline.joblib']


In [89]:
TEXT_PATH = "/content/stress_text_pipeline.joblib"

In [90]:
print(type(obj))
print("Is tuple?", isinstance(obj, tuple))

<class 'tuple'>
Is tuple? True


In [92]:
import joblib, numpy as np

TEXT_PATH = "/content/stress_text_pipeline.joblib"
obj = joblib.load(TEXT_PATH)

print("Loaded type:", type(obj), "| tuple?", isinstance(obj, tuple))

# --- Find tfidf and clf inside the tuple ---
tfidf = None
clf = None

if isinstance(obj, tuple):
    for item in obj:
        name = type(item).__name__.lower()

        # tfidf vectorizer
        if "tfidf" in name or "vectorizer" in name:
            tfidf = item

        # classifier (logreg/svm/etc.)
        if hasattr(item, "predict_proba") or hasattr(item, "predict"):
            clf = item
else:
    # if it's already a pipeline, keep it
    if hasattr(obj, "predict_proba"):
        clf = obj

print("tfidf:", type(tfidf))
print("clf  :", type(clf))

if tfidf is None or clf is None:
    raise ValueError("Could not find BOTH tfidf and clf inside stress_text_pipeline.joblib")

# --- score function (higher = more stress) ---
def text_score(text: str) -> float:
    X = tfidf.transform([text])          # 2D sparse matrix
    proba = clf.predict_proba(X)[0]      # [p(class0), p(class1)]
    # by default we assume class1 = stress
    return float(proba[1])

# --- test quickly ---
calm = "Today was calm and I feel relaxed."
panic = "I can't sleep, my heart is racing, and I'm panicking."

print("classes_:", getattr(clf, "classes_", None))
print("calm :", text_score(calm))
print("panic:", text_score(panic))

Loaded type: <class 'tuple'> | tuple? True
tfidf: <class 'sklearn.feature_extraction.text.TfidfVectorizer'>
clf  : <class 'sklearn.linear_model._logistic.LogisticRegression'>
classes_: [0 1]
calm : 0.6738594689603875
panic: 0.567139897049096


In [101]:
def text_score(text: str) -> float:
    X = tfidf.transform([text])
    proba = clf.predict_proba(X)[0]
    return float(1.0 - proba[1])   # invert

In [102]:
print("LOW :", text_score("Everything is peaceful. I feel fine and relaxed."))
print("MOD :", text_score("I feel some pressure from work but I can manage."))
print("HIGH:", text_score("I feel terrified and panicked. I can't cope anymore."))

LOW : 0.42639191002035004
MOD : 0.25679636330852273
HIGH: 0.30436702654052883


In [103]:
def text_score(text: str) -> float:
    X = tfidf.transform([text])
    proba = clf.predict_proba(X)[0]

    # invert because class1 = NOT stress
    return float(1.0 - proba[1])

In [104]:
print("LOW :", text_score("Everything is peaceful. I feel fine and relaxed."))
print("MOD :", text_score("I feel some pressure from work but I can manage."))
print("HIGH:", text_score("I feel terrified and panicked. I can't cope anymore."))

LOW : 0.42639191002035004
MOD : 0.25679636330852273
HIGH: 0.30436702654052883


In [105]:
def stress_label(score):
    if score < 0.30:
        return "LOW"
    elif score < 0.45:
        return "MODERATE"
    else:
        return "HIGH"

In [107]:
print(clf.classes_)

[0 1]


In [108]:
print(text_score("I feel great today"))
print(text_score("I am extremely stressed and overwhelmed"))

0.43031429881961414
0.4301167596316273


In [109]:
import re

STRESS_WORDS = [
    "stress", "stressed", "overwhelmed", "anxious", "anxiety", "panic", "panicking",
    "can't sleep", "insomnia", "heart racing", "chest tight", "terrified", "fear",
    "hopeless", "breakdown", "can't cope", "helpless", "crying", "pressure"
]
CALM_WORDS = [
    "calm", "relaxed", "peaceful", "fine", "okay", "stable", "comfortable",
    "under control", "confident", "happy", "good", "great"
]

def text_score_rule(text: str) -> float:
    t = text.lower()
    # simple phrase matching
    stress_hits = sum(1 for w in STRESS_WORDS if w in t)
    calm_hits   = sum(1 for w in CALM_WORDS if w in t)

    # normalize to 0..1
    raw = stress_hits - calm_hits

    # squish into probability-like range
    # raw <= -2 -> ~0.1, raw=0 -> 0.5, raw>=+2 -> ~0.9
    score = 1 / (1 + (2.71828 ** (-1.2 * raw)))
    return float(score)

In [110]:
print("great:", text_score_rule("I feel great today"))
print("stress:", text_score_rule("I am extremely stressed and overwhelmed"))
print("panic:", text_score_rule("I can't sleep, my heart is racing and I'm panicking"))
print("calm :", text_score_rule("Everything is peaceful. I feel fine and relaxed"))

great: 0.2314753600944283
stress: 0.9734029437301228
panic: 0.9734029437301228
calm : 0.026597056269877323


In [111]:
def stress_label(score):
    if score < 0.30:
        return "LOW"
    elif score < 0.45:
        return "MODERATE"
    else:
        return "HIGH"

In [112]:
def stress_label(score):
    if score < 0.30:
        return "LOW"
    elif score < 0.45:
        return "MODERATE"
    else:
        return "HIGH"

In [113]:
print(text_score_rule("I feel relaxed and everything is under control"))
print(text_score_rule("Today was a peaceful day and I feel fine"))
print(text_score_rule("I am happy and comfortable with my work"))
print(text_score_rule("Things are going well and I feel stable"))
print(text_score_rule("I feel calm and confident about tomorrow"))

0.0831728195975229
0.0831728195975229
0.0831728195975229
0.2314753600944283
0.0831728195975229


In [114]:
print(text_score_rule("I feel relaxed and everything is under control"))
print(text_score_rule("Today was a peaceful day and I feel fine"))
print(text_score_rule("I am happy and comfortable with my work"))
print(text_score_rule("Things are going well and I feel stable"))
print(text_score_rule("I feel calm and confident about tomorrow"))

0.0831728195975229
0.0831728195975229
0.0831728195975229
0.2314753600944283
0.0831728195975229


In [115]:
print(text_score_rule("Work has been a bit stressful lately"))
print(text_score_rule("I feel some pressure from my responsibilities"))
print(text_score_rule("Today was tiring and mentally exhausting"))
print(text_score_rule("I feel worried about my upcoming deadlines"))
print(text_score_rule("There is a lot on my mind right now"))

0.7685246399055716
0.7685246399055716
0.5
0.5
0.5


In [120]:
import joblib

audio_pipe = joblib.load("stress_audio_pipeline.joblib")
scaler = audio_pipe.named_steps["scaler"] if hasattr(audio_pipe, "named_steps") else None

print("Audio pipeline type:", type(audio_pipe))
print("Has named_steps:", hasattr(audio_pipe, "named_steps"))
print("Expected n_features:", scaler.n_features_in_ if scaler is not None else "unknown")

Audio pipeline type: <class 'sklearn.pipeline.Pipeline'>
Has named_steps: True
Expected n_features: 94


In [121]:
import numpy as np
import joblib

audio_pipe = joblib.load("stress_audio_pipeline.joblib")
EXPECTED_AUDIO_FEATS = audio_pipe.named_steps["scaler"].n_features_in_

def fix_len(x: np.ndarray, n: int) -> np.ndarray:
    x = np.asarray(x).astype(np.float32).flatten()
    if len(x) == n:
        return x
    if len(x) > n:
        return x[:n]
    # pad with zeros if shorter
    return np.pad(x, (0, n - len(x)), mode="constant")

def audio_score(path: str) -> float:
    # your existing extractor (keep it exactly as you have)
    x = extract_audio_features(path)          # <-- whatever you already use
    x = fix_len(x, EXPECTED_AUDIO_FEATS)      # <-- THIS is the key line
    X = x.reshape(1, -1)

    proba = audio_pipe.predict_proba(X)[0]
    # binary classifier case: take prob of class 1
    if len(proba) == 2:
        return float(proba[1])
    return float(np.max(proba))

In [123]:
import glob, os, random

audio_files = sorted(glob.glob("audio/*.wav"))
video_files = sorted(glob.glob("video/*.avi"))

print("Audio count:", len(audio_files))
print("Video count:", len(video_files))

print("\nAudio sample:", [os.path.basename(x) for x in audio_files[:5]])
print("Video sample:", [os.path.basename(x) for x in video_files[:5]])

# pick one real file from each
a = random.choice(audio_files)
v = random.choice(video_files)

print("\nChosen audio:", a)
print("Chosen video:", v)

res = predict_fused(
    "I feel stressed, but i am okay.",
    a,
    v
)
print("\nResult:", res)

Audio count: 61
Video count: 103

Audio sample: ['03-01-01-01-01-01-01.wav', '03-01-01-01-01-02-01.wav', '03-01-01-01-02-01-01.wav', '03-01-01-01-02-02-01.wav', '03-01-02-01-01-01-01.wav']
Video sample: ['01-01-01-01-01-01-01.avi', '01-01-01-01-02-01-01.avi', '01-01-01-01-02-02-01.avi', '01-01-02-01-01-01-01.avi', '01-01-02-01-01-02-01.avi']

Chosen audio: audio/03-01-06-01-01-02-01.wav
Chosen video: video/01-01-02-02-01-01-01.avi

Result: {'p_text': 0.3637739290544201, 'p_audio': 0.8850395988394122, 'p_video': 0.5404897928237915, 'p_final': 0.5471051056314795, 'final_level': 'Moderate'}
